# Creating a dataset
### The aim of this notebook is to construct a dataset containing country-level aggregates of IHD (Ischemic Heart Disease) incident rates, combined with all other relevant indicators. The final output dataset is expected to have the following structure:
- **col_name**: country
- **description**: Name of the country from OECD member and partner countries.
- **type**: string
- **data_source**: OECD 
- **col_name**: year
- **description**: Year of observation.
- **type**: int (values: 2000-2018)
- **data_source**: OECD
- **col_name**: gender
- **description**: Sex of the population group.
- **type**: string (values: Female, Male)
- **data_source**: OECD
- **col_name**: population
- **description**: Absolute number of people in the specific group (country, year, gender).
- **type**: float
- **data_source**: OECD
- **col_name**: num_of_incidence
- **description**: Absolute number of new Ischemic Heart Disease (IHD) cases in the group.
- **type**: float
- **data_source**: IHME
- **col_name**: per_of_incidence
- **description**: Share of this specific group's (country/year/gender) IHD cases in the global total IHD cases.
- **type**: float
- **data_source**: IHME
- **col_name**: incidence_rate
- **description**: Number of new cases per 100,000 people.
- **type**: float
- **data_source**: custom column
- **col_name**: weight_35f
- **description**: Coefficient prepared for use in the model to simulate recruitment with a 35% Female and 65% Male participation rate.
- **type**: float
- **data_source**: custom column
- **col_name**: weight_30f
- **description**: Coefficient prepared for use in the model to simulate recruitment with a 30% Female and 70% Male participation rate.
- **type**: float
- **data_source**: custom column
- **col_name**: weight_25f
- **description**: Coefficient prepared for use in the model to simulate recruitment with a 25% Female and 75% Male participation rate.
- **type**: float
- **data_source**: custom column
- **col_name**: alcohol_consumption
- **description**: Average annual alcohol consumption per person aged 15+ (liters of pure alcohol).
- **type**: float
- **data_source**: OECD
- **col_name**: cholesterol_level
- **description**: Mean total cholesterol level in adults age-standardized (mmol/L).
- **type**: float
- **data_source**: WHO
- **col_name**: diabetes_level
- **description**: Prevalence of diabetes in adults aged 18 and above (age-standardized, %).
- **type**: float
- **data_source**: WHO
- **col_name**: hypertension_level
- **description**: Prevalence of raised blood pressure in adults aged 30–79 (age-standardized, %).
- **type**: float
- **data_source**: WHO
- **col_name**: smoking_level
- **description**: Share of population aged 15+ who are daily smokers (age-standardized, %).
- **type**: float
- **data_source**: OECD
- **col_name**: obesity_level
- **description**: Share of population who are obese (BMI ≥ 30 kg/m²), based on measured and self-reported data (age-standardized, %).
- **type**: float
- **data_source**: OECD
- **col_name**: health_expenditure
- **description**: Current health expenditure as percentage of GDP.
- **type**: float
- **data_source**: WHO
- **col_name**: country_year_gender
- **description**: Unique ID, which combines country, year, and gender information.
- **type**: string
- **data_source**: custom column

In [7]:
# Importing libraries
#!pip install scikit-learn
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer

In [ ]:
#### Popultion Data
#loading data
population_data = pd.read_csv("C:/Users/user/Desktop/magisterka/sem_3/KU/data collection/population.csv")

#selecting valid columns and renaming them
population_data = population_data[["Reference area", "Sex", "TIME_PERIOD", "OBS_VALUE"]].rename(columns = {
    "Reference area" : "country",
    "Sex" : "gender",
    "TIME_PERIOD" : "year",
    "OBS_VALUE" : "population"
})

#droping missing values
population_data = population_data.dropna()

#creating a key column
population_data["country_year_gender"] = population_data["country"] + "_" + population_data["year"].astype(str) + "_" + population_data["gender"]

#droping duplicates
population_data = population_data.drop_duplicates(subset=["country_year_gender"])


#### Number of new cases of IHD 
##female data
number_of_incidence_female = pd.read_csv("C:/Users/user/Desktop/magisterka/sem_3/KU/data collection/number_of_new_incidence_female.csv")


#selecting valid columns and renaming them
number_of_incidence_female = number_of_incidence_female[["Location", "Sex", "Year", "Value"]].rename(columns = {
    "Location" : "country",
    "Sex" : "gender",
    "Year" : "year",
    "Value" : "num_of_incidence"
})

#droping rows with missing value in incident rate column
number_of_incidence_female = number_of_incidence_female.dropna()

#changing type of year column
number_of_incidence_female['year'] = number_of_incidence_female['year'].astype('int')

#creating a key column
number_of_incidence_female["country_year_gender"] = number_of_incidence_female["country"] + "_" + number_of_incidence_female["year"].astype(str) + "_" + number_of_incidence_female["gender"]

#droping duplicates
number_of_incidence_female = number_of_incidence_female.drop_duplicates(subset=["country_year_gender"])

#### Number of new cases of IHD 
##male data
number_of_incidence_male = pd.read_csv("C:/Users/user/Desktop/magisterka/sem_3/KU/data collection/number_of_new_incidence_male.csv")


#selecting valid columns and renaming them
number_of_incidence_male = number_of_incidence_male[["Location", "Sex", "Year", "Value"]].rename(columns = {
    "Location" : "country",
    "Sex" : "gender",
    "Year" : "year",
    "Value" : "num_of_incidence"
})

#droping rows with missing value in incident rate column
number_of_incidence_male = number_of_incidence_male.dropna()

#changing type of year column
number_of_incidence_male['year'] = number_of_incidence_male['year'].astype('int')

#creating a key column
number_of_incidence_male["country_year_gender"] = number_of_incidence_male["country"] + "_" + number_of_incidence_male["year"].astype(str) + "_" + number_of_incidence_male["gender"]

#droping duplicates
number_of_incidence_male = number_of_incidence_male.drop_duplicates(subset=["country_year_gender"])

number_of_incidence_male.head()

#### Combining male and female 
number_of_incidence_data = pd.concat(
    [number_of_incidence_female, number_of_incidence_male],
    ignore_index=True
)


#### number of new cases of IHD in %
##female data
per_of_incidence_female = pd.read_csv("C:/Users/user/Desktop/magisterka/sem_3/KU/data collection/per_of_new_incidence_female.csv")


#selecting valid columns and renaming them
per_of_incidence_female = per_of_incidence_female[["Location", "Sex", "Year", "Value"]].rename(columns = {
    "Location" : "country",
    "Sex" : "gender",
    "Year" : "year",
    "Value" : "per_of_incidence"
})

#droping rows with missing value in incident rate column
per_of_incidence_female = per_of_incidence_female.dropna()

#changing type of year column
per_of_incidence_female['year'] = per_of_incidence_female['year'].astype('int')

#creating a key column
per_of_incidence_female["country_year_gender"] = per_of_incidence_female["country"] + "_" + per_of_incidence_female["year"].astype(str) + "_" + per_of_incidence_female["gender"]

#droping duplicates
per_of_incidence_female = per_of_incidence_female.drop_duplicates(subset=["country_year_gender"])

#### per of new cases of IHD 
##male data
per_of_incidence_male = pd.read_csv("C:/Users/user/Desktop/magisterka/sem_3/KU/data collection/per_of_new_incidence_male.csv")


#selecting valid columns and renaming them
per_of_incidence_male = per_of_incidence_male[["Location", "Sex", "Year", "Value"]].rename(columns = {
    "Location" : "country",
    "Sex" : "gender",
    "Year" : "year",
    "Value" : "per_of_incidence"
})

#droping rows with missing value in incident rate column
per_of_incidence_male = per_of_incidence_male.dropna()

#changing type of year column
per_of_incidence_male['year'] = per_of_incidence_male['year'].astype('int')

#creating a key column
per_of_incidence_male["country_year_gender"] = per_of_incidence_male["country"] + "_" + per_of_incidence_male["year"].astype(str) + "_" + per_of_incidence_male["gender"]

#droping duplicates
per_of_incidence_male = per_of_incidence_male.drop_duplicates(subset=["country_year_gender"])

per_of_incidence_male.head()

#### Combining male and female 
per_of_incidence_data = pd.concat(
    [per_of_incidence_female, per_of_incidence_male],
    ignore_index=True
)

#### Alcohol consumption Data
#loading the data
alcohol_consumption_data = pd.read_csv("C:/Users/user/Desktop/magisterka/sem_3/KU/data collection/alcohol_consumption.csv")

#selecting and renaming columns
alcohol_consumption_data = alcohol_consumption_data[["Reference area", "TIME_PERIOD", "OBS_VALUE"]].rename(columns = {
    "Reference area" : "country",
    "TIME_PERIOD" : "year",
    "OBS_VALUE" : "alcohol_consumption"
})

#creating a column gender
male = alcohol_consumption_data.copy()
male["gender"] = "Male"

female = alcohol_consumption_data.copy()
female["gender"] = "Female"

alcohol_consumption_data = pd.concat([male, female], ignore_index=True)

#droping missing values
alcohol_consumption_data = alcohol_consumption_data.dropna()

#creating a key column
alcohol_consumption_data["country_year_gender"] = alcohol_consumption_data["country"] + "_" + alcohol_consumption_data["year"].astype(str) + "_" + alcohol_consumption_data["gender"]

#droping duplicates
alcohol_consumption_data = alcohol_consumption_data.drop_duplicates(subset=["country_year_gender"])

#### Cholesterol Data
#loading the data
cholesterol_data = pd.read_csv("C:/Users/user/Desktop/magisterka/sem_3/KU/data collection/cholesterol.csv")

#picking valid column and renaming them
cholesterol_data = cholesterol_data[["Location", "Dim1", "Period", "FactValueNumeric"]].rename(columns = {
    "Location" : "country",
    "Dim1" : "gender",
    "Period" : "year",
    "FactValueNumeric" :"cholesterol_level"
})

#droping missing values
cholesterol_data = cholesterol_data.dropna()

#droping gender = Both sexes
cholesterol_data = cholesterol_data[cholesterol_data["gender"] != "Both sexes"]

#creating a key column
cholesterol_data["country_year_gender"] = cholesterol_data["country"] + "_" + cholesterol_data["year"].astype(str) + "_" + cholesterol_data["gender"]

#droping duplicates
cholesterol_data = cholesterol_data.drop_duplicates(subset=["country_year_gender"])

#cholesterol_data.head()

#### Diabetes Data
#loading data
diabetes_data = pd.read_csv("C:/Users/user/Desktop/magisterka/sem_3/KU/data collection/diabetes.csv")

#picking valid column and renaming them
diabetes_data = diabetes_data[["Location", "Dim1", "Period", "FactValueNumeric"]].rename(columns = {
    "Location" : "country",
    "Dim1" : "gender",
    "Period" : "year",
    "FactValueNumeric" :"diabetes_level"
})

#droping missing values
diabetes_data = diabetes_data.dropna()

#creating a key column
diabetes_data["country_year_gender"] = diabetes_data["country"] + "_" + diabetes_data["year"].astype(str) + "_" + diabetes_data["gender"]

#droping duplicates
diabetes_data = diabetes_data.drop_duplicates(subset=["country_year_gender"])

#diabetes_data.head()

#### Hypertension Data
#loading data
hypertension_data = pd.read_csv("C:/Users/user/Desktop/magisterka/sem_3/KU/data collection/hypertension.csv")

#picking valid column and renaming them
hypertension_data = hypertension_data[["Location", "Dim1", "Period", "FactValueNumeric"]].rename(columns = {
    "Location" : "country",
    "Dim1" : "gender",
    "Period" : "year",
    "FactValueNumeric" :"hypertension_level"
})

#droping missing values
hypertension_data = hypertension_data.dropna()

#droping gender = Both sexes
hypertension_data = hypertension_data[hypertension_data["gender"] != "Both sexes"]

#creating a key column
hypertension_data["country_year_gender"] = hypertension_data["country"] + "_" + hypertension_data["year"].astype(str) + "_" + hypertension_data["gender"]

#droping duplicates
hypertension_data = hypertension_data.drop_duplicates(subset=["country_year_gender"])

#hypertension_data.head()


#### Obesity Data
#loading data
obesity_data = pd.read_csv("C:/Users/user/Desktop/magisterka/sem_3/KU/data collection/obesity.csv")

#picking valid column and renaming them
obesity_data = obesity_data[["Reference area", "Sex", "TIME_PERIOD", "OBS_VALUE"]].rename(columns = {
    "Reference area" : "country",
    "Sex" : "gender",
    "TIME_PERIOD" : "year",
    "OBS_VALUE" : "obesity_level"
})

#droping missing values
obesity_data = obesity_data.dropna()

#creating a key column
obesity_data["country_year_gender"] = obesity_data["country"] + "_" + obesity_data["year"].astype(str) + "_" + obesity_data["gender"]

#droping duplicates
obesity_data = obesity_data.drop_duplicates(subset=["country_year_gender"])

#### Smoking Data
#loading data
smoking_data = pd.read_csv("C:/Users/user/Desktop/magisterka/sem_3/KU/data collection/smoking.csv")

#picking valid column and renaming them
smoking_data = smoking_data[["Reference area", "Sex", "TIME_PERIOD", "OBS_VALUE"]].rename(columns = {
    "Reference area" : "country",
    "Sex" : "gender",
    "TIME_PERIOD" : "year",
    "OBS_VALUE" : "smoking_level"
})

#droping missing values
smoking_data = smoking_data.dropna()

#creating a key column
smoking_data["country_year_gender"] = smoking_data["country"] + "_" + smoking_data["year"].astype(str) + "_" + smoking_data["gender"]

#droping duplicates
smoking_data = smoking_data.drop_duplicates(subset=["country_year_gender"])

#### Health Expenditure Data
#loading the data
health_expenditure = pd.read_csv("C:/Users/user/Desktop/magisterka/sem_3/KU/data collection/health_expenditure.csv")

#selecting and renaming columns
health_expenditure = health_expenditure[["Location", "Period", "Value"]].rename(columns = {
    "Location" : "country",
    "Period" : "year",
    "Value" : "health_expenditure"
})

#creating a column gender
male = health_expenditure.copy()
male["gender"] = "Male"

female = health_expenditure.copy()
female["gender"] = "Female"

health_expenditure_data = pd.concat([male, female], ignore_index=True)

#droping missing values
health_expenditure_data = health_expenditure_data.dropna()

#creating a key column
health_expenditure_data["country_year_gender"] = health_expenditure_data["country"] + "_" + health_expenditure_data["year"].astype(str) + "_" + health_expenditure_data["gender"]

#droping duplicates
health_expenditure_data = health_expenditure_data.drop_duplicates(subset=["country_year_gender"])

In [9]:
#checkind date ranges for all df
datasets = {
    "Number of new IHD cases": number_of_incidence_data,
    "% of new IHD cases" : per_of_incidence_data,
    "Population" : population_data,    
    "Alcohol": alcohol_consumption_data,
    "Cholesterol": cholesterol_data,
    "Diabetes": diabetes_data,
    "Hypertension": hypertension_data,
    "Obesity": obesity_data,
    "Smoking": smoking_data,
    "Health Expenditure": health_expenditure_data  
}

print("YEAR RANGE FOR EACH DATASET")
for name, df in datasets.items():
    if 'year' in df.columns:
        years = sorted(df["year"].dropna().unique())
        print(f"{name:20} | Years: {years[0]} – {years[-1]} | Number of years: {len(years)}")
    else:
        print(f"{name:20} | MISSING 'year' column")

#based on the output the data range will be limited to 2000-2018

YEAR RANGE FOR EACH DATASET
Number of new IHD cases | Years: 1990 – 2023 | Number of years: 34
% of new IHD cases   | Years: 1990 – 2023 | Number of years: 34
Population           | Years: 1950 – 2024 | Number of years: 75
Alcohol              | Years: 1960 – 2024 | Number of years: 65
Cholesterol          | Years: 1980 – 2018 | Number of years: 39
Diabetes             | Years: 1990 – 2021 | Number of years: 32
Hypertension         | Years: 1990 – 2019 | Number of years: 30
Obesity              | Years: 1975 – 2024 | Number of years: 50
Smoking              | Years: 1960 – 2024 | Number of years: 65
Health Expenditure   | Years: 2000 – 2022 | Number of years: 23


In [10]:
#limiting the time range
start_year = 2000
end_year = 2018

def limit_years(df, start=start_year, end=end_year):
    if 'year' in df.columns:
        before = len(df)
        df = df[(df["year"] >= start) & (df["year"] <= end)].copy()
        return df
    else:
        return df
        
number_of_incidence_data = limit_years(number_of_incidence_data)
per_of_incidence_data = limit_years(per_of_incidence_data)
population_data = limit_years(population_data)
alcohol_consumption_data = limit_years(alcohol_consumption_data)
cholesterol_data = limit_years(cholesterol_data)
diabetes_data = limit_years(diabetes_data)
hypertension_data = limit_years(hypertension_data)
obesity_data = limit_years(obesity_data)
smoking_data = limit_years(smoking_data)
health_expenditure_data = limit_years(health_expenditure_data)



# left join 
full_data = population_data.copy()

#datasets to join
datasets_to_merge = [
    number_of_incidence_data[["country_year_gender", "num_of_incidence"]],
    per_of_incidence_data[["country_year_gender", "per_of_incidence"]],
    alcohol_consumption_data[["country_year_gender", "alcohol_consumption"]],
    cholesterol_data[["country_year_gender", "cholesterol_level"]],
    diabetes_data[["country_year_gender", "diabetes_level"]],
    hypertension_data[["country_year_gender", "hypertension_level"]],
    obesity_data[["country_year_gender", "obesity_level"]],
    smoking_data[["country_year_gender", "smoking_level"]],
    health_expenditure_data[["country_year_gender", "health_expenditure"]]
]

#joining all datasets
for ds in datasets_to_merge:
    full_data = full_data.merge(ds, on="country_year_gender", how="left")

print(f"After merge: {len(full_data)} rows")


#coverage check 
feature_cols = [col for col in full_data.columns if col not in ["country", "year", "gender", "country_year_gender", "mortality_rate"]]
coverage = full_data[feature_cols].notna().mean() * 100
print("Feature Coverage (%)")
print(coverage.sort_values())

After merge: 1444 rows
Feature Coverage (%)
obesity_level           45.290859
smoking_level           53.324100
cholesterol_level       86.842105
hypertension_level      86.842105
health_expenditure      86.842105
diabetes_level          86.842105
per_of_incidence        92.105263
num_of_incidence        92.105263
population             100.000000
alcohol_consumption    100.000000
dtype: float64


In [13]:
#creating a dataset without the pre-processing steps and without alcohol_consumption, health_expenditure columns

#droping the alcohol_consumption, health_expenditure columns
dataset_no_preprocessing = full_data.drop(columns=["alcohol_consumption", "health_expenditure"])

#adding 3 weight columns for different gender bias scenarios

incidence_by_gender = dataset_no_preprocessing.groupby('gender')['num_of_incidence'].sum()

total_incidence = incidence_by_gender.sum()

#actual Proportion (P_Actual)
P_Actual_F = incidence_by_gender.get('Female', 0) / total_incidence
P_Actual_M = incidence_by_gender.get('Male', 0) / total_incidence

print(f"Actual Female Proportion (P_Actual_F): {P_Actual_F:.4f}")
print(f"Actual Male Proportion (P_Actual_M): {P_Actual_M:.4f}")

#defining female proportions for weighting
target_proportions_f = {
    'weight_35f': 0.35,   # 35% Female, 65% Male
    'weight_30f': 0.30,   # 30% Female, 70% Male
    'weight_25f': 0.25    # 25% Female, 75% Male
}

#dictionary to hold the calculated weights (W_F and W_M) for each scenario
weights_to_map = {}

for col_name, P_Target_F in target_proportions_f.items():
    #target Proportion for Males
    P_Target_M = 1.0 - P_Target_F
    
    #calculate Weights: W = P_Target / P_Actual
    W_F = P_Target_F / P_Actual_F
    W_M = P_Target_M / P_Actual_M
    
    #store the weights for mapping (Female -> W_F, Male -> W_M)
    weights_to_map[col_name] = {
        'Female': W_F,
        'Male': W_M
    }
    
    print(f"\nScenario {col_name} (F:{P_Target_F*100:.0f}%, M:{P_Target_M*100:.0f}%):")
    print(f"  Female Weight (W_F): {W_F:.4f}")
    print(f"  Male Weight (W_M): {W_M:.4f}")


#adding weight columns to the dataset

for col_name, weight_map in weights_to_map.items():
    # Map the calculated weights to the corresponding sex for each observation
    dataset_no_preprocessing[col_name] = dataset_no_preprocessing['gender'].map(weight_map)

dataset_no_preprocessing.head()

#adding incidence_rate column
dataset_no_preprocessing['incidence_rate'] = (dataset_no_preprocessing['num_of_incidence'] / dataset_no_preprocessing['population']) * 100000

#organizing dataset
desired_order = [
    "country",
    "year",
    "gender",
    "population",
    "num_of_incidence",
    "per_of_incidence",
    "incidence_rate",
    "weight_35f",
    "weight_30f",
    "weight_25f",
    "cholesterol_level",
    "diabetes_level",
    "hypertension_level",
    "smoking_level",
    "obesity_level",
    "country_year_gender"
]

dataset_no_preprocessing = dataset_no_preprocessing[desired_order]

#saving dataset
dataset_no_preprocessing.to_csv("C:/Users/user/Desktop/magisterka/sem_3/KU/data collection/final_ihd_dataset_without_prepocessing.csv", index=False)


Actual Female Proportion (P_Actual_F): 0.3856
Actual Male Proportion (P_Actual_M): 0.6144

Scenario weight_35f (F:35%, M:65%):
  Female Weight (W_F): 0.9076
  Male Weight (W_M): 1.0580

Scenario weight_30f (F:30%, M:70%):
  Female Weight (W_F): 0.7780
  Male Weight (W_M): 1.1394

Scenario weight_25f (F:25%, M:75%):
  Female Weight (W_F): 0.6483
  Male Weight (W_M): 1.2207


In [14]:
#2nd dataset for the project with pre-processing steps

#replacing NaN values for all the variables expect of alcohol consumption, per_of_incidence and num_of_incidence
feature_cols = [
    "num_of_incidence",
    "per_of_incidence",
    "alcohol_consumption",   
    "cholesterol_level",
    "diabetes_level",
    "hypertension_level",
    "smoking_level",
    "health_expenditure",
    "obesity_level"          
]

exclude_from_imputation = [
    "alcohol_consumption",
    "per_of_incidence",
    "num_of_incidence"
]



impute_cols = [
    col for col in feature_cols 
    if col not in exclude_from_imputation 
]

if impute_cols:
    print(f"\nImputing {len(impute_cols)} features: {impute_cols}")
    context_cols = ["year", "alcohol_consumption"]  
    context_cols += [c for c in feature_cols if c not in impute_cols]
    
    # Add gender as context
    full_data["is_female"] = (full_data["gender"] == "Female").astype(int)
    context_cols += ["is_female"]
    
    # Full list for scaling
    impute_with_context = impute_cols + context_cols
    
    # Scale
    scaler = StandardScaler()
    scaled = scaler.fit_transform(full_data[impute_with_context])
    
    # KNN
    imputer = KNNImputer(n_neighbors=5, weights="distance")
    imputed = imputer.fit_transform(scaled)
    
    # Restore only imputed columns
    imputed_df = pd.DataFrame(
        scaler.inverse_transform(imputed),
        columns=impute_with_context,
        index=full_data.index
    )
    full_data[impute_cols] = imputed_df[impute_cols]
    
    # Clean up
    full_data = full_data.drop(columns=["is_female"], errors='ignore')
else:
    print("\nNo variables need imputation.")


#final coverage check
final_coverage = full_data[feature_cols].notna().mean() * 100
print("FINAL COVERAGE (AFTER IMPUTATION)")
print(final_coverage.round(2))


Imputing 6 features: ['cholesterol_level', 'diabetes_level', 'hypertension_level', 'smoking_level', 'health_expenditure', 'obesity_level']
FINAL COVERAGE (AFTER IMPUTATION)
num_of_incidence        92.11
per_of_incidence        92.11
alcohol_consumption    100.00
cholesterol_level      100.00
diabetes_level         100.00
hypertension_level     100.00
smoking_level          100.00
health_expenditure     100.00
obesity_level          100.00
dtype: float64


In [15]:
# Adding 3 weight columns for different gender bias scenarios

incidence_by_gender = full_data.groupby('gender')['num_of_incidence'].sum()

total_incidence = incidence_by_gender.sum()

# Actual Proportion (P_Actual)
P_Actual_F = incidence_by_gender.get('Female', 0) / total_incidence
P_Actual_M = incidence_by_gender.get('Male', 0) / total_incidence

print(f"Actual Female Proportion (P_Actual_F): {P_Actual_F:.4f}")
print(f"Actual Male Proportion (P_Actual_M): {P_Actual_M:.4f}")

#defining female proportions for weighting
target_proportions_f = {
    'weight_35f': 0.35,   # 35% Female, 65% Male
    'weight_30f': 0.30,   # 30% Female, 70% Male
    'weight_25f': 0.25    # 25% Female, 75% Male
}

# Dictionary to hold the calculated weights (W_F and W_M) for each scenario
weights_to_map = {}

for col_name, P_Target_F in target_proportions_f.items():
    # Target Proportion for Males
    P_Target_M = 1.0 - P_Target_F
    
    # Calculate Weights: W = P_Target / P_Actual
    W_F = P_Target_F / P_Actual_F
    W_M = P_Target_M / P_Actual_M
    
    # Store the weights for mapping (Female -> W_F, Male -> W_M)
    weights_to_map[col_name] = {
        'Female': W_F,
        'Male': W_M
    }
    
    print(f"\nScenario {col_name} (F:{P_Target_F*100:.0f}%, M:{P_Target_M*100:.0f}%):")
    print(f"  Female Weight (W_F): {W_F:.4f}")
    print(f"  Male Weight (W_M): {W_M:.4f}")


#adding weight columns to the dataset

for col_name, weight_map in weights_to_map.items():
    # Map the calculated weights to the corresponding sex for each observation
    full_data[col_name] = full_data['gender'].map(weight_map)

full_data.head()

Actual Female Proportion (P_Actual_F): 0.3856
Actual Male Proportion (P_Actual_M): 0.6144

Scenario weight_35f (F:35%, M:65%):
  Female Weight (W_F): 0.9076
  Male Weight (W_M): 1.0580

Scenario weight_30f (F:30%, M:70%):
  Female Weight (W_F): 0.7780
  Male Weight (W_M): 1.1394

Scenario weight_25f (F:25%, M:75%):
  Female Weight (W_F): 0.6483
  Male Weight (W_M): 1.2207


,country,gender,year,population,country_year_gender,num_of_incidence,per_of_incidence,alcohol_consumption,cholesterol_level,diabetes_level,hypertension_level,obesity_level,smoking_level,health_expenditure,weight_35f,weight_30f,weight_25f
0,Belgium,Male,2000,5012015.0,Belgium_2000_Male,24704.480129,0.001402,11.6,5.3,4.02,41.4,11.568701,31.172316,8.00,1.057978,1.139361,1.220744
1,Belgium,Male,2001,5030157.0,Belgium_2001_Male,25231.798641,0.001420,11.5,5.2,4.07,41.5,11.500000,28.600000,8.15,1.057978,1.139361,1.220744
2,Belgium,Male,2002,5054585.0,Belgium_2002_Male,25762.820018,0.001437,11.4,5.2,4.13,41.6,13.159581,29.597734,8.32,1.057978,1.139361,1.220744
3,Belgium,Male,2003,5077032.0,Belgium_2003_Male,26244.033172,0.001450,11.4,5.1,4.20,41.7,13.661866,28.375992,9.16,1.057978,1.139361,1.220744
4,Belgium,Male,2004,5099248.0,Belgium_2004_Male,26596.806319,0.001455,11.1,5.1,4.27,41.7,11.900000,28.000000,9.33,1.057978,1.139361,1.220744


In [16]:
#### Calculating the output variable Y which is the IHD new cases rate
full_data['incidence_rate'] = (full_data['num_of_incidence'] / full_data['population']) * 100000

#changing the order of columns
desired_order = [
    "country",
    "year",
    "gender",
    "population",
    "num_of_incidence",
    "per_of_incidence",
    "incidence_rate",
    "weight_35f",
    "weight_30f",
    "weight_25f",
    "alcohol_consumption",
    "cholesterol_level",
    "diabetes_level",
    "hypertension_level",
    "smoking_level",
    "obesity_level",
    "health_expenditure",
    "country_year_gender"
]

full_data = full_data[desired_order]


#saving the dataset as csv file
full_data.to_csv("C:/Users/user/Desktop/magisterka/sem_3/KU/data collection/final_ihd_dataset_with_prepocessing.csv", index=False)